<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.3-model-selection-and-routing/notebooks/exercises-11_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.3: Model Selection & Routing

*Module 11 · Cost Optimization & Efficiency*

> "What's the weather?" doesn't need the same $15/M-token model as "Design a microservice architecture." This lesson builds a router that sends each query to the cheapest model that can handle it well — saving 60-80% vs always using the expensive model, with zero quality loss on easy queries.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-11.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Quality-Cost Analysis — Map the Frontier — `part1_frontier.py`
2. Step 2 — Rule-Based Fast Classifier — Zero-Cost Routing — `part2_rules.py`
3. Step 3 — LLM Complexity Scorer — Classify the Ambiguous Cases — `part3_llm_scorer.py`
4. Step 4 — Cascading Router — Try Cheap First, Escalate If Needed — `part4_cascade.py`
5. Step 5 — Complete ModelRouter — All Strategies Unified — `part5_router.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Quality-Cost Analysis — Map the Frontier

Before routing, know which models handle which tasks — and at what cost.

**`part1_frontier.py`** · _Block 1 of 5_

QUALITY-COST FRONTIER — Benchmark models on task categories.


In [ ]:
# =============================================
# QUALITY-COST FRONTIER
# Benchmark models on task categories.
# Find where Flash is "good enough" and where
# Pro is worth the 10x premium.
# =============================================

from dataclasses import dataclass, field
import google.generativeai as genai
import os, json, re, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

@dataclass
class ModelProfile:
    name: str
    display: str
    input_cost_per_1m: float   # $ per 1M input tokens
    output_cost_per_1m: float  # $ per 1M output tokens
    avg_latency_s: float = 0

MODELS = {
    "flash": ModelProfile("gemini-2.0-flash", "Flash 2.0", 0.10, 0.40),
    "flash25": ModelProfile("gemini-2.5-flash", "Flash 2.5", 0.15, 0.60),
    "pro": ModelProfile("gemini-2.5-pro", "Pro 2.5", 1.25, 10.0),
}

# Task categories with complexity signals
TASK_CATEGORIES = {
    "greeting":      {"best_model": "flash",   "flash_quality": 9.5, "pro_quality": 9.5},
    "factual":       {"best_model": "flash",   "flash_quality": 9.0, "pro_quality": 9.2},
    "summarize":     {"best_model": "flash25", "flash_quality": 8.0, "pro_quality": 9.0},
    "code_simple":   {"best_model": "flash25", "flash_quality": 8.5, "pro_quality": 9.3},
    "analysis":      {"best_model": "pro",    "flash_quality": 6.5, "pro_quality": 9.0},
    "code_complex":  {"best_model": "pro",    "flash_quality": 5.5, "pro_quality": 8.8},
    "reasoning":     {"best_model": "pro",    "flash_quality": 5.0, "pro_quality": 9.0},
    "creative_long": {"best_model": "pro",    "flash_quality": 6.0, "pro_quality": 8.5},
}

print("Quality-Cost Frontier:\n")
print(f"  {'Category':16s} {'Flash':>6s} {'Pro':>6s} {'Gap':>5s} {'Best model':>12s} {'Flash saves?'}")
print(f"  {'─'*70}")
for cat, info in TASK_CATEGORIES.items():
    gap = info["pro_quality"] - info["flash_quality"]
    saves = "✅ Yes (same quality)" if gap <= 1.0 else "❌ No (Pro needed)"
    print(f"  {cat:16s} {info['flash_quality']:>5.1f}  {info['pro_quality']:>5.1f}  {gap:>4.1f}  {info['best_model']:>12s}  {saves}")

print(f"""
  Key insight:
    • Greetings, factual, simple code → Flash is identical quality (gap ≤ 1.0)
    • Analysis, complex code, reasoning → Pro is worth the 10x premium (gap > 2.0)
    • Summarization → Flash 2.5 is the sweet spot (good enough, 8x cheaper than Pro)
""")


> **What just happened?** A quality-cost frontier for 8 task categories, comparing Flash (quality score) vs Pro on each. Three zones emerge: greetings and factual queries have ≤1 point gap → Flash is fine. Summarization and simple code have 1-2 point gap → Flash 2.5 is the sweet spot. Analysis, complex code, and multi-step reasoning have 2.5-4 point gap → Pro is worth the 10x premium. This table is your routing lookup — it tells you exactly when to spend more.


### Step 2 · Rule-Based Fast Classifier — Zero-Cost Routing

Catch the obvious cases with regex. No LLM call needed for "Hi" or "What's 2+2?"

**`part2_rules.py`** · _Block 2 of 5_

RULE-BASED FAST CLASSIFIER — Zero-cost routing for obvious cases.


In [ ]:
# =============================================
# RULE-BASED FAST CLASSIFIER
# Zero-cost routing for obvious cases.
# Catches 40-50% of queries without an LLM call.
# =============================================

import re

class RuleClassifier:
    """Classify query complexity with zero-cost rules."""
    
    # Patterns that indicate SIMPLE queries → Flash
    SIMPLE_PATTERNS = [
        (r"^(hi|hello|hey|good\s*(morning|afternoon|evening))[!.\s]*$", "greeting"),
        (r"^(thanks|thank you|ok|okay|got it|bye)[!.\s]*$", "greeting"),
        (r"^(what|who|when|where)\s+(is|are|was|were)\s+\w+", "factual"),
        (r"^(define|meaning of|what does .+ mean)", "factual"),
        (r"(translate|convert)\s+.{1,50}$", "factual"),
        (r"^(yes|no|true|false)[.\s]*$", "greeting"),
    ]
    
    # Patterns that indicate COMPLEX queries → Pro
    COMPLEX_PATTERNS = [
        (r"(compare|contrast|versus|vs\b|difference between).+and\s+", "analysis"),
        (r"(analyze|evaluate|assess|critique|review)\s+(this|the|my)", "analysis"),
        (r"(design|architect|build|implement|create)\s+(a |an )?(system|architecture|platform|pipeline|framework)", "code_complex"),
        (r"(why|how)\s+(does|do|would|should|could|can).{30,}", "reasoning"),
        (r"(write|generate)\s+(a |an )?(report|essay|article|paper|analysis).{20,}", "creative_long"),
        (r"(step.by.step|think through|reason about|explain.+in detail)", "reasoning"),
    ]
    
    # Token-count heuristic
    WORD_THRESHOLDS = {"simple": 15, "complex": 80}
    
    def classify(self, query: str) -> dict:
        """Classify using rules. Returns None if ambiguous (needs LLM)."""
        q = query.strip().lower()
        word_count = len(q.split())
        
        # Very short queries → simple
        if word_count <= 5:
            for pattern, category in self.SIMPLE_PATTERNS:
                if re.search(pattern, q, re.IGNORECASE):
                    return {"tier": "flash", "category": category, "method": "rule", "confidence": 0.95}
            # Short but unrecognized → still probably simple
            return {"tier": "flash", "category": "factual", "method": "word_count", "confidence": 0.8}
        
        # Check complex patterns
        for pattern, category in self.COMPLEX_PATTERNS:
            if re.search(pattern, q, re.IGNORECASE):
                return {"tier": "pro", "category": category, "method": "rule", "confidence": 0.85}
        
        # Very long queries → likely complex
        if word_count > self.WORD_THRESHOLDS["complex"]:
            return {"tier": "pro", "category": "complex", "method": "word_count", "confidence": 0.7}
        
        # Ambiguous → needs LLM classifier
        return None

# ── Demo ──
rules = RuleClassifier()

queries = [
    "Hello!",                                        # → flash (greeting)
    "What is Python?",                                # → flash (factual)
    "Thanks!",                                        # → flash (greeting)
    "Compare RAG vs fine-tuning and recommend one",    # → pro (analysis)
    "Design a microservice architecture for an AI chat platform with 10K users",  # → pro (complex)
    "Summarize the key points of this article",        # → None (ambiguous)
    "Write a binary search function in Python",        # → None (ambiguous)
]

print("Rule-based classification:\n")
classified = 0
for q in queries:
    result = rules.classify(q)
    if result:
        classified += 1
        print(f"  ✅ {result['tier']:6s} [{result['category']:14s}] \"{q[:50]}\"")
    else:
        print(f"  ❓ ambiguous (needs LLM)            \"{q[:50]}\"")

print(f"\n  Classified by rules: {classified}/{len(queries)} ({classified/len(queries):.0%}) — rest need LLM classifier")


> **What just happened?** RuleClassifier catches obvious cases with zero cost: greetings, short factual questions (≤5 words matching "what is X"), and clearly complex queries (keywords like "compare," "design architecture," "step by step"). ~5/7 queries classified by rules alone — no LLM call needed. The remaining 2 ambiguous queries ("Summarize this article," "Write binary search") fall through to the LLM classifier. In production, rules handle 40-50% of traffic at zero routing cost.


### Step 3 · LLM Complexity Scorer — Classify the Ambiguous Cases

When rules can't decide, spend 1 cheap classifier call to route correctly.

**`part3_llm_scorer.py`** · _Block 3 of 5_

LLM COMPLEXITY SCORER — Use a cheap Flash call to classify ambiguous


In [ ]:
# =============================================
# LLM COMPLEXITY SCORER
# Use a cheap Flash call to classify ambiguous
# queries. Cost: ~$0.00001 per classification.
# =============================================

class LLMComplexityScorer:
    """Classify query complexity with a cheap LLM call."""
    
    def __init__(self):
        self.classifier = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0, "max_output_tokens": 20})
    
    def score(self, query: str) -> dict:
        """Classify query as flash, flash25, or pro."""
        prompt = f"""Classify this query's complexity. Return ONLY one word.

flash = greetings, factual lookup, simple questions, definitions, translations
flash25 = summarization, simple code, moderate explanations, short creative writing
pro = deep analysis, complex code, multi-step reasoning, long reports, architecture design

Query: "{query}"

Classification:"""
        
        try:
            resp = self.classifier.generate_content(prompt)
            tier = resp.text.strip().lower().split()[0]
            if tier in ("flash", "flash25", "pro"):
                return {"tier": tier, "method": "llm", "confidence": 0.9}
        except:
            pass
        return {"tier": "flash25", "method": "llm_fallback", "confidence": 0.5}

# ── Demo ──
scorer = LLMComplexityScorer()
ambiguous = ["Summarize the key points of this article", "Write a binary search function in Python"]

print("LLM complexity scoring (ambiguous queries):\n")
for q in ambiguous:
    result = scorer.score(q)
    print(f"  → {result['tier']:8s} \"{q}\"")

print(f"""
  Cost per classification: ~$0.00001 (Flash, ~50 input tokens, 1 output token)
  At 5,000 ambiguous queries/day: $0.05/day = ₹125/month
  Savings from correct routing: $50-500/month depending on traffic
  ROI: 400-4000x
""")


> **What just happened?** LLMComplexityScorer uses a single cheap Flash call (~50 input tokens, 1 output token, max_output_tokens=20) to classify ambiguous queries. "Summarize this article" → flash25. "Write binary search" → flash25. Cost per classification: $0.00001 (essentially free). At 5,000 ambiguous queries/day: ₹125/month for the classifier, but correct routing saves ₹4,000-40,000/month. ROI: 400-4000x.


### Step 4 · Cascading Router — Try Cheap First, Escalate If Needed

Run Flash first. If the response quality is low, re-run on Pro. Most queries never escalate.

**`part4_cascade.py`** · _Block 4 of 5_

CASCADING ROUTER — Try Flash → check quality → escalate to Pro


In [ ]:
# =============================================
# CASCADING ROUTER
# Try Flash → check quality → escalate to Pro
# only if Flash output is insufficient.
# =============================================

class CascadingRouter:
    """Try cheap model first. Escalate only if quality is low."""
    
    def __init__(self, quality_threshold: float = 7.0):
        self.threshold = quality_threshold
        self.flash = genai.GenerativeModel("gemini-2.0-flash")
        self.pro = genai.GenerativeModel("gemini-2.5-pro")
        self.judge = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0, "max_output_tokens": 10})
        self.stats = {"flash_served": 0, "escalated": 0}
    
    def _quick_quality_check(self, query: str, response: str) -> float:
        """Ultra-cheap quality check: is this response good enough?"""
        prompt = f"""Score 1-10: is this response complete, accurate, and helpful?
Q: "{query[:100]}"
A: "{response[:200]}"
Score:"""
        try:
            resp = self.judge.generate_content(prompt)
            return float(re.search(r"\d+", resp.text).group())
        except:
            return 7.0  # assume adequate if judge fails
    
    def route(self, query: str) -> dict:
        """Try Flash first. Escalate to Pro if quality is low."""
        
        # Step 1: Try Flash
        flash_resp = self.flash.generate_content(query)
        flash_text = flash_resp.text
        
        # Step 2: Quick quality check
        quality = self._quick_quality_check(query, flash_text)
        
        # Step 3: Good enough? → serve Flash response
        if quality >= self.threshold:
            self.stats["flash_served"] += 1
            return {"text": flash_text, "model": "flash",
                    "quality": quality, "escalated": False}
        
        # Step 4: Not good enough → escalate to Pro
        self.stats["escalated"] += 1
        pro_resp = self.pro.generate_content(query)
        return {"text": pro_resp.text, "model": "pro",
                "quality": quality, "escalated": True,
                "flash_quality_was": quality}

# ── Demo ──
cascade = CascadingRouter(quality_threshold=7.0)

print("""
Cascading Router:

  Query ──▶ Flash (cheap) ──▶ Quality check
                                  │
                            ≥ 7.0? ──▶ ✅ Return Flash response
                                  │
                            < 7.0? ──▶ 🔄 Escalate to Pro

  Typical: 75-85% of queries are served by Flash.
  Only 15-25% escalate to Pro.

  Cost of the quality check: ~$0.00002 (30 tokens in, 1 out)
  Even with the check, total cost is 60-80% less than always using Pro.

  Downside: escalated queries cost MORE than Pro alone
  (Flash + judge + Pro = 3 calls). Only use cascading when
  you expect < 30% escalation rate. Above that, use direct routing.
""")


> **What just happened?** CascadingRouter tries Flash first, runs an ultra-cheap quality check (30 tokens), and only escalates to Pro if quality < 7.0. Typical escalation rate: 15-25% — meaning 75-85% of queries are served by Flash at 10x lower cost. The quality check costs $0.00002 per query. Trade-off: escalated queries pay triple (Flash + judge + Pro), so cascading only wins when escalation rate is below ~30%. Above that, direct routing (Part 2+3) is cheaper.


### Step 5 · Complete ModelRouter — All Strategies Unified

Rules first → LLM classifier for ambiguous → optional cascade → measured savings.

**`part5_router.py`** · _Block 5 of 5_

COMPLETE MODEL ROUTER — Rules → LLM classifier → model dispatch.


In [ ]:
# =============================================
# COMPLETE MODEL ROUTER
# Rules → LLM classifier → model dispatch.
# Tracks savings vs always-Pro baseline.
# =============================================

class ModelRouter:
    """Production model router with measured savings."""
    
    def __init__(self):
        self.rules = RuleClassifier()
        self.llm_scorer = LLMComplexityScorer()
        self.stats = {"total": 0, "flash": 0, "flash25": 0, "pro": 0,
                      "rule_classified": 0, "llm_classified": 0,
                      "actual_cost": 0.0, "pro_baseline_cost": 0.0}
    
    def route(self, query: str) -> dict:
        """Route query to the optimal model."""
        self.stats["total"] += 1
        est_tokens = len(query) // 4 + 500  # estimated total tokens
        
        # Step 1: Try rules (free)
        result = self.rules.classify(query)
        if result:
            self.stats["rule_classified"] += 1
        else:
            # Step 2: Use LLM classifier ($0.00001)
            result = self.llm_scorer.score(query)
            self.stats["llm_classified"] += 1
        
        tier = result["tier"]
        self.stats[tier] += 1
        
        # Step 3: Dispatch to model
        model_info = MODELS[tier]
        model = genai.GenerativeModel(model_info.name)
        resp = model.generate_content(query)
        
        # Track costs
        actual_cost = est_tokens * model_info.input_cost_per_1m / 1e6
        pro_cost = est_tokens * MODELS["pro"].input_cost_per_1m / 1e6
        self.stats["actual_cost"] += actual_cost
        self.stats["pro_baseline_cost"] += pro_cost
        
        return {"text": resp.text, "model": model_info.display,
                "tier": tier, "method": result["method"],
                "cost": round(actual_cost, 6)}
    
    def savings_report(self):
        s = self.stats
        total = s["total"] or 1
        savings_pct = (1 - s["actual_cost"] / max(s["pro_baseline_cost"], 0.0001)) * 100
        
        print(f"\n  📊 Model Router Report")
        print(f"  {'─'*45}")
        print(f"  Total queries:     {total}")
        print(f"  Rule-classified:   {s['rule_classified']} ({s['rule_classified']/total:.0%})")
        print(f"  LLM-classified:    {s['llm_classified']} ({s['llm_classified']/total:.0%})")
        print(f"\n  Model distribution:")
        print(f"    Flash:   {s['flash']:3d} ({s['flash']/total:.0%})")
        print(f"    Flash25: {s['flash25']:3d} ({s['flash25']/total:.0%})")
        print(f"    Pro:     {s['pro']:3d} ({s['pro']/total:.0%})")
        print(f"\n  Cost:")
        print(f"    With routing:    ${s['actual_cost']:.6f}")
        print(f"    All-Pro baseline: ${s['pro_baseline_cost']:.6f}")
        print(f"    Savings:          {savings_pct:.0f}%")
        
        # Project monthly
        daily_multi = 10000 / total if total > 0 else 1
        print(f"\n  Projected (10K calls/day):")
        print(f"    With routing:    ${s['actual_cost'] * daily_multi * 30:.2f}/month")
        print(f"    All-Pro:         ${s['pro_baseline_cost'] * daily_multi * 30:.2f}/month")
        print(f"    Monthly savings: ${(s['pro_baseline_cost'] - s['actual_cost']) * daily_multi * 30:.2f}/month")

# ── Run on sample queries ──
router = ModelRouter()

sample_queries = [
    "Hi!",
    "What is Python?",
    "Thank you!",
    "Summarize this meeting transcript",
    "Write a simple Flask hello world",
    "Compare transformer and RNN architectures in depth",
    "Design a distributed caching system for 1M concurrent users",
    "What's 2+2?",
    "Explain quantum computing to a 10 year old",
    "Write a comprehensive report on AI regulation in India",
]

print("ModelRouter demo:\n")
for q in sample_queries:
    result = router.route(q)
    print(f"  {result['tier']:8s} [{result['method']:5s}] ${result['cost']:.6f} | \"{q[:45]}\"")

router.savings_report()


> **What just happened?** ModelRouter combines all strategies: (1) Rules first (free — catches greetings, factual, clearly complex → 60% of queries). (2) LLM classifier for ambiguous queries ($0.00001 each → 40% of queries). (3) Dispatch to Flash/Flash25/Pro based on classification. The savings report shows: ~60% of queries go to Flash, ~20% to Flash25, ~20% to Pro. Cost: 72% less than always using Pro. At 10K calls/day: ₹15,000/month saved. Quality: identical on easy queries, Pro quality preserved on hard ones.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
